# Linear Regression
**IFRI AI Classes**

---

Linear Regression is one of the oldest and most widely used algorithms in machine learning and statistics. It models the relationship between one or more input features and a continuous target variable by fitting a linear equation to observed data. Despite its simplicity, it remains a strong baseline for many real-world prediction tasks and a building block for more complex models.

The `ifri-mini-ml-lib` implementation supports both **simple linear regression** (one feature) and **multiple linear regression** (several features), with two optimization strategies: the closed-form **Ordinary Least Squares** and iterative **Gradient Descent**.

## 1. Key Concepts

### Simple Linear Regression

Simple linear regression models the relationship between a single feature $x$ and a continuous target $y$ using a straight line:

$$\hat{y} = w \cdot x + b$$

Where:
- $\hat{y}$ is the predicted value
- $w$ is the **slope** (coefficient/weight)
- $b$ is the **intercept** (bias)

### Multiple Linear Regression

When there are $n$ features, the model generalizes to:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b = \mathbf{w}^\top \mathbf{x} + b$$

In matrix form, for a dataset of $m$ samples:

$$\hat{\mathbf{y}} = X\mathbf{w} + b$$

Where $X \in \mathbb{R}^{m \times n}$ is the feature matrix.

### Loss Function: Mean Squared Error (MSE)

Training a linear regression model means finding the values of $\mathbf{w}$ and $b$ that minimize the **Mean Squared Error** between predictions and true values:

$$\mathcal{L}(\mathbf{w}, b) = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)^2$$

This loss function is convex, which guarantees that any local minimum is also a global minimum.

## 2. Optimization Methods

The `LinearRegression` class in `ifri-mini-ml-lib` supports two methods to minimize the MSE loss.

### Ordinary Least Squares (OLS) — `method='least_squares'`

OLS derives the optimal parameters analytically, without iteration.

**For simple regression**, the closed-form solution is:

$$w = \frac{\sum_{i=1}^{m}(x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{m}(x_i - \bar{x})^2}, \quad b = \bar{y} - w\bar{x}$$

**For multiple regression**, using the augmented feature matrix $\tilde{X} = [\mathbf{1} \mid X]$:

$$\boldsymbol{\theta} = (\tilde{X}^\top \tilde{X})^{+} \tilde{X}^\top \mathbf{y}$$

Where $(\cdot)^{+}$ denotes the **Moore-Penrose pseudoinverse**, which handles cases where $\tilde{X}^\top \tilde{X}$ is singular.

> **Advantage**: Exact solution in a single step.  
> **Limitation**: Computationally expensive for very large datasets ($O(n^3)$ matrix inversion).

---

### Gradient Descent — `method='gradient_descent'`

Gradient Descent is an iterative optimization method that updates the parameters in the direction of the steepest descent of the loss function.

At each iteration $t$, the gradients of the MSE with respect to $\mathbf{w}$ and $b$ are:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = -\frac{2}{m} X^\top (\mathbf{y} - \hat{\mathbf{y}}), \quad \frac{\partial \mathcal{L}}{\partial b} = -\frac{2}{m} \sum_{i=1}^{m}(y_i - \hat{y}_i)$$

The parameters are then updated as:

$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{w}}, \quad b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

Where $\alpha$ is the **learning rate** — a hyperparameter controlling the step size.

> **Advantage**: Scales well to large datasets and high-dimensional feature spaces.  
>**Limitation**: Requires careful tuning of `learning_rate` and `epochs`; may not converge if $\alpha$ is too large.

---

> **Note**: For most standard datasets, `least_squares` is the recommended default. Switch to `gradient_descent` when your dataset is very large or the feature matrix is ill-conditioned.

## 3. Pseudo-algorithm

```
Input: X       ← matrice [n_samples, n_features]
       y       ← vecteur cible [n_samples]
       method  ← "least_squares" ou "gradient_descent"
       α       ← learning rate (si gradient_descent)
       epochs  ← nombre d'itérations (si gradient_descent)

if method == "least_squares":
    if n_features == 1:
        X ← flatten(X)
        w ← sum((X - mean(X)) * (y - mean(y))) / sum((X - mean(X))²)
        b ← mean(y) - w * mean(X)
    else:
        X̃ ← [1 | X]                    # Ajout colonne de biais
        θ ← pinv(X̃ᵀ X̃) · X̃ᵀ · y      # Pseudoinverse
        b ← θ[0],  w ← θ[1:]

elif method == "gradient_descent":
    if n_features == 1:
        w ← 0.0,  b ← 0.0
    else:
        w ← zeros(n_features),  b ← 0.0
    for t = 1 to epochs do
        ŷ  ← X · w + b
        dw ← -(2/m) · Xᵀ · (y - ŷ)
        db ← -(2/m) · sum(y - ŷ)
        w  ← w - α · dw
        b  ← b - α · db
    end for

Return w, b
```

## 4. Implementation

For this implementation, we use the **California Housing** dataset, which contains housing data for California districts. The goal is to predict the median house value (`MedHouseVal`) from features such as median income, average number of rooms, population, etc.

It has 20,640 samples and 8 features, making it a suitable benchmark for regression tasks.

In [7]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter


housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.Series(housing.target, name='MedHouseVal')

print(f"Dataset shape : {X.shape}")
X.head()

Dataset shape : (20640, 8)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [8]:

splitter = DataSplitter(seed=42)
X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)


### With Scikit-learn

In [9]:
import time
from sklearn.linear_model import LinearRegression as SklearnLR

start_1 = time.perf_counter()
sk_model = SklearnLR()
sk_model.fit(X_train, y_train)
end_1 = time.perf_counter()

y_pred_sk = sk_model.predict(X_test)

### With ifri-mini-ml-lib

In [10]:
from ifri_mini_ml_lib.regression import LinearRegression

# Scikit-learn
start_1 = time.perf_counter()
sk_model = SklearnLR()
sk_model.fit(X_train, y_train)
end_1 = time.perf_counter()
y_pred_sk = sk_model.predict(X_test)

# ifri-mini — Least Squares
start_2 = time.perf_counter()
lr_ols = LinearRegression(method="least_squares")
lr_ols.fit(X_train, y_train)
end_2 = time.perf_counter()
y_pred_ols = lr_ols.predict(X_test)

In [11]:
from ifri_mini_ml_lib.metrics.regression import evaluate_rg_model

m_sk  = evaluate_rg_model(y_test.tolist(), y_pred_sk)
m_ols = evaluate_rg_model(y_test.tolist(), y_pred_ols)

results = pd.DataFrame({
    'Metric': ['R²', 'RMSE', 'MAE', 'MAPE', 'Time (s)'],
    'Scikit-learn': [
        m_sk['R²'], m_sk['RMSE'], m_sk['MAE'], m_sk['MAPE'],
        round(end_1 - start_1, 6)
    ],
    'ifri-mini (OLS)': [
        m_ols['R²'], m_ols['RMSE'], m_ols['MAE'], m_ols['MAPE'],
        round(end_2 - start_2, 6)
    ],
})

results.set_index('Metric')

,Scikit-learn,ifri-mini (OLS)
Metric,,
R²,0.575788,0.575788
RMSE,0.745581,0.745581
MAE,0.533200,0.533200
MAPE,31.952187,31.952187
Time (s),0.009207,0.010678


## 5. Interactive Demo

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from ipywidgets import interact, Dropdown, FloatSlider, IntSlider
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter
from ifri_mini_ml_lib.regression.linear_regression import LinearRegression
from ifri_mini_ml_lib.metrics.regression import evaluate_rg_model

feature_names = housing.feature_names

def plot_linear_regression(feature=feature_names[0], method='least_squares', learning_rate=0.01, epochs=500):
    idx = list(feature_names).index(feature)

    x_series = pd.DataFrame(housing.data[:, idx], columns=[feature])
    y_series  = pd.Series(housing.target, name='target')

    splitter = DataSplitter(seed=42)
    x_tr, x_te, y_tr, y_te = splitter.train_test_split(x_series, y_series, test_size=0.2)
    x_tr, x_te = x_tr.values, x_te.values
    y_tr, y_te = y_tr.values, y_te.values

    # Normalisation manuelle pour éviter l'overflow en gradient descent
    x_mean, x_std = x_tr.mean(), x_tr.std() + 1e-8
    x_tr_scaled = (x_tr - x_mean) / x_std
    x_te_scaled  = (x_te - x_mean) / x_std

    model = LinearRegression(method=method, learning_rate=learning_rate, epochs=epochs)
    model.fit(x_tr_scaled, y_tr)
    y_hat = model.predict(x_te_scaled)

    metrics = evaluate_rg_model(y_te.tolist(), y_hat)

    x_line = np.linspace(x_te.min(), x_te.max(), 100).reshape(-1, 1)
    x_line_scaled = (x_line - x_mean) / x_std
    y_line = model.predict(x_line_scaled)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x_te, y_te, alpha=0.3, s=10, label='Test data')
    ax.plot(x_line, y_line, color='red', linewidth=2,
            label=f"R²={metrics['R²']:.3f}")
    ax.set_xlabel(feature)
    ax.set_ylabel('Median House Value')
    ax.set_title(f'Linear Regression — {feature} | {method}')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact(
    plot_linear_regression,
    feature=Dropdown(options=feature_names, value='MedInc', description='Feature:'),
    method=Dropdown(options=['least_squares', 'gradient_descent'], description='Method:'),
    learning_rate=FloatSlider(min=0.001, max=0.1, step=0.001, value=0.01, description='LR:'),
    epochs=IntSlider(min=100, max=2000, step=100, value=500, description='Epochs:')
);

interactive(children=(Dropdown(description='Feature:', options=('MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',…

The interactive demo above allows you to:
- Select any feature from the California Housing dataset and visualize how it correlates with house prices
- Switch between `least_squares` and `gradient_descent` to compare optimization strategies
- Tune the learning rate and number of epochs when using gradient descent, and observe its effect on the fit quality (R²)



## 6. Real-life Applications

Linear regression, despite its simplicity, powers a wide range of real-world systems.

**Real Estate and Housing Markets**: Linear regression is used to estimate property prices based on features such as location, surface area, number of rooms, and proximity to amenities. Models similar to the one above are deployed by platforms like Zillow to provide automated property valuations.

**Agriculture and Crop Yield Prediction**: In precision agriculture, linear regression models trained on soil quality, rainfall, and temperature data help predict crop yields at a regional level, supporting food security planning. This is especially relevant in West Africa, where initiatives like [FAOSTAT](https://www.fao.org/faostat) collect the kind of data such models require.

**Economics and Financial Forecasting**: Economists use multiple linear regression to model relationships between macroeconomic variables — for example, how GDP growth depends on inflation, interest rates, and government spending. Central banks and investment firms use similar approaches for short-term forecasting.

**Healthcare**: Linear regression is used to model clinical outcomes — for example, predicting a patient's hospital stay duration or medication dosage based on body weight and biological markers. It also appears in epidemiological studies linking environmental factors to disease rates.

**Energy Sector**: Energy companies apply linear regression to forecast electricity demand based on temperature, time of day, and population density. This allows better grid management and reduces waste.

## 7. Limitations and Challenges

**Linearity Assumption**: Linear regression assumes that the relationship between inputs and output is linear. When this assumption is violated — for example, when the true relationship is quadratic or exponential — the model will underfit and produce poor predictions. The `PolynomialRegression` class in `ifri-mini-ml-lib` addresses this by expanding features to higher-degree terms.

**Sensitivity to Outliers**: Because MSE squares the errors, outliers exert a disproportionately large influence on the learned parameters. A single extreme value can shift the regression line significantly. Robust alternatives (e.g., using MAE as the loss) exist but are not part of the current implementation.

**Multicollinearity**: When input features are highly correlated with each other, the OLS solution becomes unstable — small changes in the data lead to large swings in the estimated coefficients. The pseudoinverse used in `_fit_multiple` mitigates numerical failures, but the resulting coefficients may still be unreliable. Regularization techniques (Ridge, Lasso) are the standard remedy.

**Feature Scaling for Gradient Descent**: The gradient descent method is sensitive to the scale of input features. If features have very different magnitudes, convergence can be slow or unstable. Applying `StandardScaler` or `MinMaxScaler` from the preprocessing module before training with gradient descent is strongly recommended.

**No Uncertainty Quantification**: The `LinearRegression` class returns point predictions only. It does not provide confidence intervals or prediction intervals around its estimates, which limits its usefulness in decision-making contexts where uncertainty matters.

## 8. References

1. *Linear Regression*, Scikit-learn Documentation — https://scikit-learn.org/stable/modules/linear_model.html

2. Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer. Chapter 3: Linear Models for Regression.

3. Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.). Springer. — https://hastie.su.domains/ElemStatLearn/

4. *Gradient Descent for Linear Regression* — Andrew Ng, Machine Learning Specialization, Coursera — https://www.coursera.org/specializations/machine-learning-introduction

5. *California Housing Dataset*, StatLib — Pace, R. K. & Barry, R. (1997). Sparse Spatial Autoregressions. *Statistics and Probability Letters*, 33(3), 291–297.